# nb5 — Varys File Agent Multi-Provider Tests

Two sections:
- **Section 1 — Mock tests** (runs in CI, no credentials required)
- **Section 2 — Live API tests** (skipped unless provider is configured, clearly marked)

Run all Section 1 cells to verify the provider abstraction without any API calls.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from varys.agent.tool_definition import ToolDefinition
from varys.agent.tool_schemas import to_anthropic_tools, to_openai_tools, to_bedrock_tools, parse_tool_input
from varys.agent.provider_base import (
    TextDelta, ThoughtDelta, ToolCall, TurnEnd,
    ToolUseNotSupportedError, AgentProvider,
)
from varys.agent.provider_factory import (
    _check_tool_support, check_tool_support_safe,
    TOOL_INCOMPATIBLE_MODELS, AgentConfigError,
)
from varys.agent.providers.anthropic_provider import AnthropicAgentProvider
from varys.agent.providers.openai_provider import OpenAIAgentProvider
from varys.agent.providers.bedrock_provider import BedrockAgentProvider
from varys.agent.providers.ollama_provider import OllamaAgentProvider
from varys.agent.providers.azure_provider import AzureOpenAIAgentProvider

print('Imports OK')

## Section 1 — Mock Tests

### 1.1 Tool schema translation

In [ ]:
# Sample ToolDefinitions matching the Read and Glob schemas
read_def = ToolDefinition(
    name='Read',
    description='Read the contents of a file.',
    parameters={
        'type': 'object',
        'properties': {'file_path': {'type': 'string', 'description': 'Path to the file.'}},
        'required': ['file_path'],
    },
)
glob_def = ToolDefinition(
    name='Glob',
    description='Find files matching a glob pattern.',
    parameters={
        'type': 'object',
        'properties': {'pattern': {'type': 'string', 'description': 'Glob pattern.'}},
        'required': ['pattern'],
    },
)

defs = [read_def, glob_def]

# Anthropic format
a_tools = to_anthropic_tools(defs)
assert a_tools[0]['name'] == 'Read'
assert 'input_schema' in a_tools[0]
assert a_tools[0]['input_schema']['type'] == 'object'

# OpenAI format
o_tools = to_openai_tools(defs)
assert o_tools[0]['type'] == 'function'
assert o_tools[0]['function']['name'] == 'Read'
assert 'parameters' in o_tools[0]['function']

# Bedrock format
b_tools = to_bedrock_tools(defs)
assert 'tools' in b_tools
assert b_tools['tools'][0]['toolSpec']['name'] == 'Read'
assert 'inputSchema' in b_tools['tools'][0]['toolSpec']

print('✓ Tool schema translation — all 3 formats OK')

### 1.2 parse_tool_input

In [ ]:
import json

# dict passthrough
assert parse_tool_input({'file_path': 'src/utils.py'}) == {'file_path': 'src/utils.py'}

# JSON string
assert parse_tool_input(json.dumps({'file_path': 'src/utils.py'})) == {'file_path': 'src/utils.py'}

# Invalid type
try:
    parse_tool_input(123)
    assert False, 'Should have raised'
except ValueError:
    pass

print('✓ parse_tool_input OK')

### 1.3 Tool support detection — proactive

In [ ]:
# Known incompatible
for provider, model in [
    ('ollama', 'llava'),
    ('ollama', 'llava:latest'),
    ('ollama', 'nomic-embed-text'),
    ('openai', 'text-embedding-3-small'),
    ('bedrock', 'amazon.titan-text-express-v1'),
    ('azure', 'text-embedding-ada-002'),
]:
    try:
        _check_tool_support(provider, model)
        assert False, f'Should have raised for {provider}/{model}'
    except ToolUseNotSupportedError:
        pass

# Known compatible — must NOT raise
for provider, model in [
    ('ollama', 'qwen2.5-coder'),
    ('ollama', 'llama3.1'),
    ('openai', 'gpt-4o'),
    ('anthropic', 'claude-3-5-sonnet-20241022'),
    ('bedrock', 'anthropic.claude-3-5-sonnet-20241022-v2:0'),
]:
    _check_tool_support(provider, model)  # should not raise

# check_tool_support_safe returns dict, never raises
r = check_tool_support_safe('ollama', 'llava')
assert r['supported'] is False
assert r['reason'] is not None

r2 = check_tool_support_safe('ollama', 'qwen2.5-coder')
assert r2['supported'] is True

print('✓ Proactive tool support detection OK')

### 1.4 History message format — all providers

In [ ]:
# Stub provider instances (no real credentials needed for history methods)
anthropic_prov = AnthropicAgentProvider(api_key='test', model='claude-3-5-sonnet-20241022')

# Mock an OpenAI-compatible client so the constructor works
class _FakeClient: pass
openai_prov = OpenAIAgentProvider(client=_FakeClient(), model='gpt-4o')
ollama_prov = OllamaAgentProvider.__new__(OllamaAgentProvider)
ollama_prov._client = _FakeClient()
ollama_prov._model = 'qwen2.5-coder'
ollama_prov._ollama_model = 'qwen2.5-coder'

tc = ToolCall(call_id='call_abc123', tool_name='Read', tool_input={'file_path': 'src/utils.py'})
text = 'I will read the file.'

# Anthropic
a_msg = anthropic_prov.build_assistant_history_message(text, [tc])
assert a_msg['role'] == 'assistant'
assert a_msg['content'][0] == {'type': 'text', 'text': text}
assert a_msg['content'][1]['type'] == 'tool_use'
assert a_msg['content'][1]['id'] == 'call_abc123'

# Anthropic tool result
a_res = anthropic_prov.build_tool_result_message([tc], ['file content here'])
assert isinstance(a_res, dict)
assert a_res['role'] == 'user'
assert a_res['content'][0]['type'] == 'tool_result'

# OpenAI
o_msg = openai_prov.build_assistant_history_message(text, [tc])
assert o_msg['role'] == 'assistant'
assert o_msg['content'] == text
assert o_msg['tool_calls'][0]['type'] == 'function'

# OpenAI tool result — list
o_res = openai_prov.build_tool_result_message([tc], ['file content here'])
assert isinstance(o_res, list)
assert o_res[0]['role'] == 'tool'
assert o_res[0]['tool_call_id'] == 'call_abc123'

print('✓ History message format — Anthropic and OpenAI/Ollama OK')

### 1.5 Bedrock history message format

In [ ]:
# BedrockAgentProvider history methods (no boto3 call needed)
bedrock_prov = BedrockAgentProvider.__new__(BedrockAgentProvider)
bedrock_prov._model_id = 'anthropic.claude-3-5-sonnet-20241022-v2:0'

tc = ToolCall(call_id='toolUse-001', tool_name='Read', tool_input={'file_path': 'src/utils.py'})
text = 'Reading file...'

b_msg = bedrock_prov.build_assistant_history_message(text, [tc])
assert b_msg['role'] == 'assistant'
assert b_msg['content'][0] == {'text': text}
assert 'toolUse' in b_msg['content'][1]
assert b_msg['content'][1]['toolUse']['toolUseId'] == 'toolUse-001'

b_res = bedrock_prov.build_tool_result_message([tc], ['file content'])
assert isinstance(b_res, dict)
assert b_res['role'] == 'user'
assert 'toolResult' in b_res['content'][0]

# make_initial_messages override
msgs = bedrock_prov.make_initial_messages('Hello')
assert msgs[0]['content'] == [{'text': 'Hello'}]

print('✓ Bedrock history message format OK')

### 1.6 ToolUseNotSupportedError fields

In [ ]:
err = ToolUseNotSupportedError(
    provider='ollama',
    model='llava',
    reason='Vision-only model does not support tool calling.',
    suggestion='Switch to qwen2.5-coder.',
)
assert err.provider == 'ollama'
assert err.model == 'llava'
assert 'llava' in str(err)
assert 'ollama' in str(err)
print('✓ ToolUseNotSupportedError fields OK')

### 1.7 Reactive detection — mock stream (OpenAI)

In [ ]:
import asyncio

# Build a fake OpenAI streaming client that returns finish_reason='stop' with no tool calls
class FakeChunk:
    class _Delta:
        content = 'Sure, I can help.'
        tool_calls = None
    class _Choice:
        delta = FakeChunk._Delta()
        finish_reason = 'stop'
    choices = [_Choice()]

class FakeStream:
    def __init__(self, chunks): self._chunks = chunks
    def __aiter__(self): return self
    async def __anext__(self):
        if self._chunks:
            return self._chunks.pop(0)
        raise StopAsyncIteration

class FakeChatCompletions:
    async def create(self, **kwargs): return FakeStream([FakeChunk()])

class FakeOpenAIClient:
    class chat:
        completions = FakeChatCompletions()

prov = OpenAIAgentProvider(client=FakeOpenAIClient(), model='gpt-4o')

# require_tool_use=False — should NOT raise (valid for /file_agent_find)
async def test_no_require():
    events = [e async for e in prov.stream_turn(
        [{'role': 'user', 'content': 'List files'}],
        [read_def], 'You are an agent.', 1024, require_tool_use=False
    )]
    return events

events = asyncio.get_event_loop().run_until_complete(test_no_require())
assert any(isinstance(e, TurnEnd) and e.stop_reason == 'end_turn' for e in events)
print('✓ Reactive detection: require_tool_use=False does not raise when no tool calls')

# require_tool_use=True — should raise ToolUseNotSupportedError
async def test_require():
    try:
        prov2 = OpenAIAgentProvider(client=FakeOpenAIClient(), model='gpt-4o')
        async for _ in prov2.stream_turn(
            [{'role': 'user', 'content': 'Edit file'}],
            [read_def], 'You are an agent.', 1024, require_tool_use=True
        ):
            pass
        return False  # should not reach here
    except ToolUseNotSupportedError:
        return True

raised = asyncio.get_event_loop().run_until_complete(test_require())
assert raised
print('✓ Reactive detection: require_tool_use=True raises when no tool calls')

### 1.8 Ollama synthetic call_id generation

In [ ]:
from varys.agent.providers.ollama_provider import OllamaAgentProvider

# Patch stream_turn on the parent so we can yield a ToolCall with empty id
async def _fake_stream(self, *args, **kwargs):
    yield ToolCall(call_id='', tool_name='Read', tool_input={'file_path': 'x.py'})
    yield TurnEnd(stop_reason='tool_use')

import varys.agent.providers.openai_provider as _openai_mod
original = _openai_mod.OpenAIAgentProvider.stream_turn
_openai_mod.OpenAIAgentProvider.stream_turn = _fake_stream

try:
    ollama = OllamaAgentProvider.__new__(OllamaAgentProvider)
    ollama._client = None
    ollama._model = 'qwen2.5-coder'
    ollama._ollama_model = 'qwen2.5-coder'

    async def run_ollama():
        return [e async for e in ollama.stream_turn(
            [], [read_def], 'sys', 1024
        )]

    evts = asyncio.get_event_loop().run_until_complete(run_ollama())
    tc_evts = [e for e in evts if isinstance(e, ToolCall)]
    assert len(tc_evts) == 1
    assert len(tc_evts[0].call_id) > 0, 'call_id should have been synthesised'
    print('✓ Ollama synthetic call_id OK:', tc_evts[0].call_id)
finally:
    _openai_mod.OpenAIAgentProvider.stream_turn = original

### 1.9 ALL_TOOL_DEFINITIONS round-trip

In [ ]:
from varys.agent.tools import ALL_TOOL_SCHEMAS, ALL_TOOL_DEFINITIONS

assert set(ALL_TOOL_DEFINITIONS.keys()) == set(ALL_TOOL_SCHEMAS.keys())

for name, td in ALL_TOOL_DEFINITIONS.items():
    schema = ALL_TOOL_SCHEMAS[name]
    assert td.name == schema['name']
    assert td.description == schema['description']
    assert td.parameters == schema['input_schema']

# Round-trip: to_anthropic_tools(ALL_TOOL_DEFINITIONS[...]) matches ALL_TOOL_SCHEMAS
read_td = ALL_TOOL_DEFINITIONS['Read']
a = to_anthropic_tools([read_td])[0]
assert a['name'] == ALL_TOOL_SCHEMAS['Read']['name']
assert a['input_schema'] == ALL_TOOL_SCHEMAS['Read']['input_schema']

print('✓ ALL_TOOL_DEFINITIONS round-trip OK')

## Section 2 — Live API Tests (skip unless configured)

In [ ]:
# Detect which providers are configured
configured = {}
if os.environ.get('ANTHROPIC_API_KEY') and os.environ.get('ANTHROPIC_CHAT_MODEL'):
    configured['anthropic'] = True
if os.environ.get('OPENAI_API_KEY') and os.environ.get('OPENAI_CHAT_MODEL'):
    configured['openai'] = True
if os.environ.get('OLLAMA_CHAT_MODEL'):
    configured['ollama'] = True

print('Configured providers:', list(configured.keys()) or ['none — Section 2 will be skipped'])

In [ ]:
# ⚠️  LIVE  — requires ANTHROPIC_API_KEY and ANTHROPIC_CHAT_MODEL
if 'anthropic' not in configured:
    print('SKIP — Anthropic not configured')
else:
    api_key = os.environ['ANTHROPIC_API_KEY']
    model   = os.environ['ANTHROPIC_CHAT_MODEL']
    prov = AnthropicAgentProvider(api_key=api_key, model=model)

    async def live_anthropic():
        events = []
        async for e in prov.stream_turn(
            [{'role': 'user', 'content': 'What is 2 + 2? Reply with just the number.'}],
            [],  # no tools — just text
            'You are a helpful assistant.',
            64,
        ):
            events.append(e)
        return events

    evts = asyncio.get_event_loop().run_until_complete(live_anthropic())
    text = ''.join(e.text for e in evts if isinstance(e, TextDelta))
    print(f'Anthropic live response: {text!r}')
    assert '4' in text

### Summary

In [ ]:
print('All Section 1 mock tests passed.')